# 04 CNN baseline (GPU)
ResNet50 on 5 folds x 5 label fractions (crop view), full-photo view at 100%, and the AZH published-anchor runs (3 seeds). Resumable: finished runs are skipped.

In [ ]:
import glob, os, sys
if os.path.isdir("/kaggle/input"):
    hits = glob.glob("/kaggle/input/**/src/woundvlm/__init__.py", recursive=True)
    assert hits, "attach the project dataset (lowerlimbwoundvlm) as an input"
    SRC = hits[0].rsplit("/woundvlm/", 1)[0]
else:
    SRC = os.path.abspath("../src")
sys.path.insert(0, SRC)
from woundvlm import env
E = env.setup()
P = E["project"]
print(E)

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
assert DEVICE == "cuda" or not E["in_kaggle"], "select a GPU accelerator (T4) for this notebook" 

In [ ]:
from woundvlm import experiments
experiments.run_cnn(P, 'main', device=DEVICE)

In [ ]:
import json
rows = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(P, 'outputs', 'cnn', '*.json')))]
import pandas as pd
t = pd.DataFrame([{k: r[k] for k in ('tag', 'n_fit', 'n_val', 'seconds')} | {'best_val_f1': max(h['val_macro_f1'] for h in r['history']), 'epochs': len(r['history'])} for r in rows])
t.to_csv(os.path.join(E['results'], '04_cnn_runs.csv'), index=False); t